# How the FP&A project was built, step by step

**The business question.** Atlântica Components (fictional) is ahead of budget on EBITDA for January–September 2026.
The CFO wants to know *why*, and whether it is the good kind of ahead: better trading, or luck in prices, timing
and currency that will not repeat.

**How this notebook is organised.** It follows the project in the order it was built:

| Step | What | Tool |
|---|---|---|
| 1 | Generate a double-entry ledger, budgets and a reforecast with planted stories | Python (NumPy) |
| 2 | Load the ledger and run the accounting checks | pandas |
| 3 | Model the P&L in dbt: sign convention, FX translation, plan vs actual | dbt on DuckDB |
| 4 | Answer the questions: budget vs actual, the EBITDA bridge, FX, ledger exceptions | pandas + Altair |
| 5 | Reconcile pandas with the dbt marts | pandas + DuckDB |
| 6 | Publish: web dashboard, Power BI and Qlik | JSON export, PBIP, Qlik script |

**The charts are interactive**: click an entity to drive the linked charts, hover for values. They are built with
[Altair](https://altair-viz.github.io/); open the [interactive version](https://guilhermerigonato.com/finance-fpa-analytics/notebook/)
to use them (GitHub shows notebooks without JavaScript).

**Run it:** `pip install pandas numpy altair duckdb dbt-duckdb`, then from the repository root
`python generator/generate.py` and `dbt build --profiles-dir .`, and run all cells (under a minute).

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import altair as alt

RAW, SEEDS = Path("../data/raw"), Path("../seeds")
FY, LAST_CLOSED = 2026, pd.Timestamp("2026-09-01")      # dbt vars current_fy and last_closed_month
pd.set_option("display.float_format", "{:,.0f}".format)

BLUE, ORANGE, GREEN, RED, GREY, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#d1453b", "#c9c8c3", "#52514e", "#e4e3df"

@alt.theme.register("portfolio", enable=True)
def portfolio_theme():
    axis = {"gridColor": GRID, "domainColor": MUTED, "tickColor": MUTED, "labelColor": MUTED,
            "titleColor": MUTED, "labelFontSize": 11, "titleFontSize": 11, "titleFontWeight": "normal"}
    return {"config": {"view": {"stroke": None}, "axis": axis, "legend": {"labelColor": MUTED, "titleColor": MUTED},
                       "title": {"anchor": "start", "fontSize": 13, "color": "#0b0b0b"},
                       "font": "system-ui, -apple-system, Segoe UI, sans-serif"}}

## Step 1. Generate the data

A real group ledger cannot be published, so [`generator/generate.py`](../generator/generate.py) writes one: four
entities (Portugal, Spain, France in euros; the UK in pounds), January 2024 to September 2026 closed, a
double-entry general ledger, a sales sub-ledger with units, monthly FX rates, the FY2025 and FY2026 budgets and the
October 9+3 reforecast. Eight stories are planted on purpose:

| # | Story | Where it should show |
|---|---|---|
| 1 | Sterling weakens through 2026 (budget rate 1.19 €/£) | FX translation in the UK |
| 2 | Portugal's new outbound carrier from March | Freight, ledger spikes |
| 3 | Spain's sales hires arrive months late | People (favourable), volume (unfavourable) |
| 4 | Portugal raises list prices 4% in January | Sales price |
| 5 | Tools, the lowest-margin line, grows fastest | Sales mix |
| 6 | A CloudSuite invoice posted twice in the UK in June | Duplicate-invoice check |
| 7 | France books the year-end bonus true-up in December 2025 | Spike check |
| 8 | A 6% steel surcharge on bearings from May | Unit cost |

The seed is fixed (`20261025`), so every run gives the same ledger.

In [2]:
gl = pd.read_csv(RAW / "gl_journal_lines.csv", dtype={"account_id": str}, parse_dates=["posting_date"])
plan = pd.read_csv(RAW / "plan_lines.csv", dtype={"account_id": str}, parse_dates=["month_start"])
vol = pd.read_csv(RAW / "sales_volume.csv", parse_dates=["month_start"])
fx = pd.read_csv(RAW / "fx_rates.csv", parse_dates=["month_start"])
accounts = pd.read_csv(SEEDS / "accounts.csv", dtype={"account_id": str})
entities = pd.read_csv(SEEDS / "entities.csv")
print(f"{len(gl):,} journal lines in {gl['journal_id'].nunique():,} journals · {len(plan):,} plan lines · "
      f"{len(vol):,} sales-volume rows · {len(fx)} FX rates")
gl.head(4)

81,072 journal lines in 40,536 journals · 4,536 plan lines · 1,200 sales-volume rows · 72 FX rates


,journal_id,line_number,entity_id,posting_date,account_id,cost_centre_id,product_line_id,amount_lc,source,vendor,document_ref
0,J0000001,1,PT01,2024-01-01,4000,SAL,BRG,"-27,390",AR,NaN,INV-PT01-240101-BRG
1,J0000001,2,PT01,2024-01-01,1100,NaN,NaN,"27,390",AR,NaN,INV-PT01-240101-BRG
2,J0000002,1,PT01,2024-01-01,5000,OPS,BRG,"16,966",INV,NaN,NaN
3,J0000002,2,PT01,2024-01-01,1300,NaN,NaN,"-16,966",INV,NaN,NaN


## Step 2. Load the ledger and run the accounting checks

A ledger is only usable if it obeys double entry. Before any analysis: every journal balances to zero, every
account is in the chart of accounts, every entity-month has an FX rate, and nothing is posted after the last
closed month.

In [3]:
gl["month_start"] = gl["posting_date"].dt.to_period("M").dt.to_timestamp()
gl = gl.merge(entities[["entity_id", "currency"]], on="entity_id")
journal_totals = gl.groupby("journal_id")["amount_lc"].sum().round(2)
checks = {
    "every journal balances (debits = credits)": (journal_totals == 0).all(),
    "every account is in the chart": gl["account_id"].isin(accounts["account_id"]).all(),
    "every entity-month has an FX rate": gl[["currency", "month_start"]].drop_duplicates()
        .merge(fx, on=["currency", "month_start"], how="left")["avg_rate_eur"].notna().all(),
    "nothing posted after the last closed month": (gl["month_start"] <= LAST_CLOSED).all(),
    "the whole ledger sums to zero": round(gl["amount_lc"].sum(), 2) == 0,
}
assert all(checks.values()), checks
pd.Series(checks, name="passes").to_frame()

,passes
every journal balances (debits = credits),True
every account is in the chart,True
every entity-month has an FX rate,True
nothing posted after the last closed month,True
the whole ledger sums to zero,True


## Step 3. Model the P&L in dbt

The SQL lives in [`models/`](../models). Three decisions shape every number downstream:

* **Sign convention.** The ledger stores debits positive and credits negative. The P&L facts flip the sign, so
  revenue is positive, costs are negative, and *any subtotal is a plain sum*.
* **Two translations.** Actuals are translated at the month's average rate (`amount_eur`) *and* at the budget rate
  (`amount_eur_bud_rate`). The difference between the two is the FX effect; everything else is trading.
* **One P&L mart.** `mart_pnl_monthly` puts actual, budget, the 9+3 forecast and prior year side by side per
  entity, account and month, so every view (variance, landing, bridge) reads one table.

The bridge marts (`mart_margin_pvm`, `mart_ebitda_bridge`) and the controller checks (`mart_journal_checks`) sit on
top. `dbt build` runs 67 nodes (models, seeds and tests: keys, relationships, accepted values, and a test that the
bridge steps add up from budget to actual EBITDA) with no failures.

In pandas, the same facts:

In [4]:
pl_accounts = accounts[accounts["account_type"] == "P&L"].set_index("account_id")
rates = fx.set_index(["currency", "month_start"])

def translate(frame, budget=False):
    r = rates.loc[list(zip(frame["currency"], frame["month_start"]))]
    frame = frame.copy()
    frame["amount_eur"] = frame["amount"] * (r["budget_rate_eur"].values if budget else r["avg_rate_eur"].values)
    frame["amount_eur_bud_rate"] = frame["amount"] * r["budget_rate_eur"].values
    return frame

act = gl[gl["account_id"].isin(pl_accounts.index)].copy()
act["amount"] = -act["amount_lc"]                                   # revenue +, costs -
act = translate(act.groupby(["entity_id", "currency", "account_id", "cost_centre_id", "product_line_id", "month_start"],
                            dropna=False)["amount"].sum().reset_index())

bud = plan[plan["scenario"] == f"BUD{FY}"].merge(entities[["entity_id", "currency"]], on="entity_id")
bud["amount"] = -bud["amount_lc"]
bud = translate(bud, budget=True)

ytd = lambda f: f[(f["month_start"].dt.year == FY) & (f["month_start"] <= LAST_CLOSED)]
ebitda_acc = pl_accounts.index[pl_accounts["pnl_subtotal"].isin(["Gross margin", "EBITDA"])]
e = lambda f: ytd(f[f["account_id"].isin(ebitda_acc)])
print(f"Group EBITDA Jan–Sep {FY}: €{e(act)['amount_eur'].sum() / 1e6:,.2f}m actual vs €{e(bud)['amount_eur'].sum() / 1e6:,.2f}m budget")

Group EBITDA Jan–Sep 2026: €4.41m actual vs €4.17m budget


## Step 4. Answer the questions

### Which entities are ahead, and when did it happen?

Portugal is well ahead, Spain and France behind, the UK slightly behind. The interactive view of this, linked to the
bridge, comes after the bridge is computed below.

In [5]:
names = entities.set_index("entity_id")["entity_name"]
by_ent = pd.DataFrame({"actual": e(act).groupby("entity_id")["amount_eur"].sum(),
                       "budget": e(bud).groupby("entity_id")["amount_eur"].sum()})
by_ent["variance"] = by_ent["actual"] - by_ent["budget"]
by_ent["variance_pct"] = by_ent["variance"] / by_ent["budget"]
by_ent = by_ent.assign(entity=names).reset_index()

monthly = pd.concat([
    e(act).groupby(["entity_id", "month_start"])["amount_eur"].sum().rename("value").reset_index().assign(series="Actual"),
    e(bud).groupby(["entity_id", "month_start"])["amount_eur"].sum().rename("value").reset_index().assign(series="Budget"),
])
monthly = pd.concat([monthly, monthly.groupby(["month_start", "series"])["value"].sum().reset_index().assign(entity_id="GROUP")])

ent = alt.selection_point(name="ent", fields=["entity_id"], on="click", clear="dblclick")
chosen = "(ent.entity_id && length(ent.entity_id)) ? ent.entity_id[0] : 'GROUP'"

bars = alt.Chart(by_ent).mark_bar(cornerRadius=3).encode(
    y=alt.Y("entity:N", title=None, sort="x"),
    x=alt.X("variance_pct:Q", title="EBITDA vs budget, Jan–Sep 2026", axis=alt.Axis(format="+%")),
    color=alt.condition("datum.variance < 0", alt.value(ORANGE), alt.value(BLUE)),
    opacity=alt.condition(ent, alt.value(1), alt.value(0.35)),
    tooltip=["entity", alt.Tooltip("actual:Q", format=",.0f"), alt.Tooltip("budget:Q", format=",.0f"),
             alt.Tooltip("variance:Q", format="+,.0f"), alt.Tooltip("variance_pct:Q", format="+.1%")],
).add_params(ent).properties(width=250, height=170, title="Click an entity")

lines = alt.Chart(monthly).transform_filter(f"datum.entity_id == ({chosen})").mark_line(point=True).encode(
    x=alt.X("month_start:T", title=None, axis=alt.Axis(format="%b")),
    y=alt.Y("value:Q", title="EBITDA (€)", axis=alt.Axis(format="~s")),
    color=alt.Color("series:N", scale=alt.Scale(domain=["Actual", "Budget"], range=[BLUE, GREY]), title=None),
    strokeDash=alt.StrokeDash("series:N", scale=alt.Scale(domain=["Actual", "Budget"], range=[[1, 0], [5, 4]]), legend=None),
    tooltip=[alt.Tooltip("month_start:T", format="%b %Y"), "series", alt.Tooltip("value:Q", format=",.0f")],
).properties(width=420, height=170, title=alt.Title(alt.expr(f"'Monthly EBITDA · ' + ({chosen})")))
by_ent[["entity", "actual", "budget", "variance", "variance_pct"]].style.format(
    {"actual": "{:,.0f}", "budget": "{:,.0f}", "variance": "{:+,.0f}", "variance_pct": "{:+.1%}"})

,entity,actual,budget,variance,variance_pct
0,Atlántica España,"1,194,708","1,366,294","-171,586",-12.6%
1,Atlantique France,"297,814","344,171","-46,357",-13.5%
2,Atlântica Portugal,"2,266,729","1,762,878","+503,852",+28.6%
3,Atlantica UK Ltd,"649,301","698,240","-48,939",-7.0%


### Why: the EBITDA bridge

A bridge walks from budget to actual EBITDA in steps a CFO can act on. Product margin is split into **volume, mix,
price and unit cost**; the other steps are services and discounts, freight, people, other opex, and **FX
translation**, measured separately so that currency does not hide in the trading lines. Product lines sell in very
different units (a fastener is not a pump), so volume is measured as actual units at budget price, which gives every
line a common unit:

* volume = change in sales at budget prices × the entity's budget margin rate
* mix = the same change at each line's own budget margin rate, minus the volume effect
* price = actual units × (actual price − budget price), at the budget rate
* unit cost = actual units × (budget unit cost − actual unit cost), at the budget rate

In [6]:
# Sales sub-ledger: units and revenue by scenario, translated
vs = vol.merge(entities[["entity_id", "currency"]], on="entity_id")
vs = vs[(vs["month_start"].dt.year == FY) & (vs["month_start"] <= LAST_CLOSED) & vs["scenario"].isin(["ACT", f"BUD{FY}"])]
r = rates.loc[list(zip(vs["currency"], vs["month_start"]))]
vs["rev_eur"] = vs["revenue_lc"] * np.where(vs["scenario"] == "ACT", r["avg_rate_eur"].values, r["budget_rate_eur"].values)
vs["rev_bud_rate"] = vs["revenue_lc"] * r["budget_rate_eur"].values
s = vs.groupby(["scenario", "entity_id", "product_line_id"])[["units", "rev_eur", "rev_bud_rate"]].sum()
a_, b_ = s.loc["ACT"], s.loc[f"BUD{FY}"]
cost = lambda f, col: -ytd(f[f["account_id"] == "5000"]).groupby(["entity_id", "product_line_id"])[col].sum()

l = pd.DataFrame({"act_units": a_["units"], "act_rev": a_["rev_eur"], "act_rev_bud_rate": a_["rev_bud_rate"],
                  "bud_units": b_["units"], "bud_rev": b_["rev_eur"],
                  "act_cost": cost(act, "amount_eur"), "act_cost_bud_rate": cost(act, "amount_eur_bud_rate"),
                  "bud_cost": cost(bud, "amount_eur")})
l["bud_price"] = l["bud_rev"] / l["bud_units"]
l["bud_unit_cost"] = l["bud_cost"] / l["bud_units"]
l["bud_margin_rate"] = (l["bud_rev"] - l["bud_cost"]) / l["bud_rev"]
l["act_at_bud_price"] = l["act_units"] * l["bud_price"]
g = l.groupby(level="entity_id")
l["ent_margin_rate"] = (g["bud_rev"].transform("sum") - g["bud_cost"].transform("sum")) / g["bud_rev"].transform("sum")
l["volume"] = (l["act_at_bud_price"] - l["bud_rev"]) * l["ent_margin_rate"]
l["mix"] = (l["act_at_bud_price"] - l["bud_rev"]) * (l["bud_margin_rate"] - l["ent_margin_rate"])
l["price"] = l["act_rev_bud_rate"] - l["act_at_bud_price"]
l["unit_cost"] = l["act_units"] * l["bud_unit_cost"] - l["act_cost_bud_rate"]
pvm = l.groupby(level="entity_id")[["volume", "mix", "price", "unit_cost"]].sum()

# The rest of the bridge from the P&L, trading at the budget rate
grp = pl_accounts["pnl_group"]
def var(accts):
    a = e(act)[e(act)["account_id"].isin(accts)].groupby("entity_id")["amount_eur_bud_rate"].sum()
    b = e(bud)[e(bud)["account_id"].isin(accts)].groupby("entity_id")["amount_eur"].sum()
    return a.sub(b, fill_value=0)

steps = pd.DataFrame({
    "Budget EBITDA": e(bud).groupby("entity_id")["amount_eur"].sum(),
    "Sales volume": pvm["volume"], "Sales mix": pvm["mix"], "Sales price": pvm["price"],
    "Services and discounts": var(["4100", "4900"]),
    "Unit cost": pvm["unit_cost"],
    "Freight": var(["5100", "5200"]),
    "People": var(grp.index[grp == "People"]),
    "Other opex": var(grp.index[grp.isin(["Facilities", "IT", "Commercial", "General and admin"])]),
    "FX translation": e(act).groupby("entity_id")["amount_eur"].sum() - e(act).groupby("entity_id")["amount_eur_bud_rate"].sum(),
    "Actual EBITDA": e(act).groupby("entity_id")["amount_eur"].sum(),
}).fillna(0)
steps.loc["GROUP"] = steps.sum()
inner = steps.columns[1:-1]
assert np.allclose(steps["Budget EBITDA"] + steps[inner].sum(axis=1), steps["Actual EBITDA"]), "the bridge must close"
(steps.T / 1e3).round(0).rename_axis("€ thousand")

entity_id,ES01,FR01,PT01,UK01,GROUP
€ thousand,,,,,
Budget EBITDA,"1,366",344,"1,763",698,"4,172"
Sales volume,-72,51,25,47,51
Sales mix,-41,-36,-71,-25,-173
Sales price,-10,0,862,15,868
Services and discounts,-109,14,-16,-12,-124
Unit cost,-86,-69,-115,-53,-324
Freight,8,-6,-169,-4,-171
People,143,-3,-2,3,141
Other opex,-4,2,-10,4,-8


In [7]:
# Waterfall: one row per entity and step with its start and end.
# The entity bars, the monthly lines and the waterfall share one selection, so they are drawn as one chart:
# click an entity at the top left and both the monthly lines and the bridge follow; double-click for the group.
rows = []
for ent_id, row in steps.iterrows():
    run = 0.0
    for i, (step, v) in enumerate(row.items()):
        total = step in ("Budget EBITDA", "Actual EBITDA")
        start, end = (0, v) if total else (run, run + v)
        run = v if total else run + v
        rows.append(dict(entity_id=ent_id, order=i, step=step, start=start, end=end, amount=v,
                         kind="Total" if total else ("Up" if v >= 0 else "Down")))
wf = pd.DataFrame(rows)

waterfall = alt.Chart(wf).transform_filter(f"datum.entity_id == ({chosen})").mark_bar(size=34).encode(
    x=alt.X("step:N", sort=list(steps.columns), title=None, axis=alt.Axis(labelAngle=-30, labelLimit=140)),
    y=alt.Y("start:Q", title="€", axis=alt.Axis(format="~s")), y2="end:Q",
    color=alt.Color("kind:N", scale=alt.Scale(domain=["Total", "Up", "Down"], range=[GREY, GREEN, RED]), legend=None),
    tooltip=["step", alt.Tooltip("amount:Q", format="+,.0f")],
).properties(width=700, height=280, title=alt.Title(alt.expr(f"'EBITDA bridge, budget to actual · ' + ({chosen})")))
alt.vconcat(alt.hconcat(bars, lines), waterfall).resolve_scale(color="independent")

alt.VConcatChart(...)

**Reading the bridge.** For the group, *Sales price* is the biggest positive step, and almost all of it is Portugal's
4% list-price rise (story 4). Unit cost (the steel surcharge, story 8), freight (Portugal's carrier, story 2) and mix
(Tools growing, story 5) pull the other way. Without Portugal's price rise the group would be below budget. Spain's
*People* step is favourable only because hires arrived late (story 3), while its volume and services miss.

### How much of the UK miss is currency?

The UK budget was set at 1.19 € per £. Translating UK actuals at the budget rate instead of the month's average
separates trading from currency. Hover the lines for the monthly rates.

In [8]:
gbp = fx[(fx["currency"] == "GBP") & (fx["month_start"].dt.year >= 2025)].melt(
    id_vars="month_start", value_vars=["avg_rate_eur", "budget_rate_eur"], var_name="rate", value_name="eur_per_gbp")
gbp["rate"] = gbp["rate"].map({"avg_rate_eur": "Monthly average", "budget_rate_eur": "Budget rate"})
uk = ytd(act[(act["entity_id"] == "UK01") & (act["account_id"] == "4000")])
uk_b = ytd(bud[(bud["entity_id"] == "UK01") & (bud["account_id"] == "4000")])
print(f"UK product sales vs budget: {uk['amount_eur_bud_rate'].sum() / uk_b['amount_eur'].sum() - 1:+.1%} at the budget rate, "
      f"{uk['amount_eur'].sum() / uk_b['amount_eur'].sum() - 1:+.1%} in euros at actual rates; "
      f"currency cost €{(uk['amount_eur_bud_rate'].sum() - uk['amount_eur'].sum()) / 1e3:,.0f}k of sales")
alt.Chart(gbp).mark_line(point=True).encode(
    x=alt.X("month_start:T", title=None), y=alt.Y("eur_per_gbp:Q", title="€ per £", scale=alt.Scale(zero=False)),
    color=alt.Color("rate:N", scale=alt.Scale(range=[BLUE, GREY]), title=None),
    tooltip=[alt.Tooltip("month_start:T", format="%b %Y"), "rate", alt.Tooltip("eur_per_gbp:Q", format=".3f")],
).properties(width=560, height=200, title="Sterling against the budget rate")

UK product sales vs budget: +1.6% at the budget rate, -2.0% in euros at actual rates; currency cost €362k of sales


alt.Chart(...)

### What should the controller clear before close?

Two checks a controller runs on the ledger itself:

* **Duplicate invoices:** the same entity, vendor, document reference and amount posted more than once.
* **Spikes:** an entity-account month that moved more than 25% against *both* its previous three months and the same
  month last year (and by more than 2,000), which ignores normal seasonality.

Hover a point for the detail.

In [9]:
first = gl[(gl["line_number"] == 1) & gl["document_ref"].notna() & (gl["amount_lc"] > 0)]
dups = (first.groupby(["entity_id", "account_id", "vendor", "document_ref", "amount_lc"])
             .agg(postings=("journal_id", "size"), month_start=("month_start", "min")).reset_index())
dups = dups[dups["postings"] > 1]

cost_lines = gl[gl["account_id"].isin(["5100", "5200"]) | gl["account_id"].str.startswith("6")]
m = cost_lines.groupby(["entity_id", "account_id", "month_start"])["amount_lc"].sum().mul(-1).rename("amount").reset_index()
m = m.sort_values(["entity_id", "account_id", "month_start"])
g = m.groupby(["entity_id", "account_id"])["amount"]
m["avg_3m"] = g.transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
m["same_month_ly"] = g.shift(12)
spikes = m[m["same_month_ly"].notna() & ((m["amount"] / m["avg_3m"] - 1).abs() > 0.25)
           & ((m["amount"] / m["same_month_ly"] - 1).abs() > 0.25) & ((m["amount"] - m["same_month_ly"]).abs() > 2000)].copy()
spikes["change_vs_ly"] = spikes["amount"] / spikes["same_month_ly"] - 1
spikes["account"] = spikes["account_id"].map(pl_accounts["account_name"])
print(f"{len(dups)} duplicate invoice(s): " + "; ".join(f"{r.entity_id} {r.vendor} {r.document_ref} ({r.postings} postings)" for r in dups.itertuples()))

alt.Chart(spikes).mark_circle(size=140).encode(
    x=alt.X("month_start:T", title=None),
    y=alt.Y("change_vs_ly:Q", title="Change vs same month last year", axis=alt.Axis(format="+%", values=[-0.5, 0, 0.5, 1, 2, 4, 6]), scale=alt.Scale(type="symlog")),
    color=alt.Color("entity_id:N", title="Entity", scale=alt.Scale(range=[BLUE, ORANGE, GREEN, "#9a6fd0"])),
    tooltip=["entity_id", "account", alt.Tooltip("month_start:T", format="%b %Y"),
             alt.Tooltip("change_vs_ly:Q", format="+.0%"), alt.Tooltip("amount:Q", title="Cost (local currency)", format=",.0f")],
).properties(width=560, height=220, title="Cost spikes flagged for review")

1 duplicate invoice(s): UK01 CloudSuite Ltd CLO-UK01-2606-0 (2 postings)


alt.Chart(...)

The checks find the planted items and nothing else: the CloudSuite invoice posted twice in the UK in June (and its
reversal in August shows as a spike), Portugal's outbound freight from March, and France's December 2025 bonus
true-up.

## Step 5. Reconcile pandas with the dbt marts

The bridge, the margin split and the checks were built twice, in SQL and in pandas. They must agree to the cent.

In [10]:
db = duckdb.connect("../data/fpa.duckdb", read_only=True)
m_bridge = db.sql("select entity_id, step, amount_eur from mart_ebitda_bridge").df().pivot(index="entity_id", columns="step", values="amount_eur")
m_checks = db.sql("select check_type, count(*) n from mart_journal_checks group by all").df().set_index("check_type")["n"]
db.close()

ents = ["PT01", "ES01", "FR01", "UK01"]
diff = (steps.loc[ents, m_bridge.columns] - m_bridge.loc[ents]).abs().max().max()
recon = pd.DataFrame([
    ["Bridge: largest difference on any entity and step (€)", diff, 0],
    ["Group actual EBITDA (€)", steps.loc["GROUP", "Actual EBITDA"], m_bridge["Actual EBITDA"].sum()],
    ["Group sales price effect (€)", steps.loc["GROUP", "Sales price"], m_bridge["Sales price"].sum()],
    ["UK FX translation (€)", steps.loc["UK01", "FX translation"], m_bridge.loc["UK01", "FX translation"]],
    ["Duplicate invoices", len(dups), m_checks.get("duplicate_invoice", 0)],
    ["Spikes", len(spikes), m_checks.get("spike", 0)],
], columns=["measure", "pandas", "dbt"])
recon["match"] = np.isclose(recon["pandas"].astype(float), recon["dbt"].astype(float), atol=0.01)
assert recon["match"].all(), recon
recon.style.format({"pandas": "{:,.2f}", "dbt": "{:,.2f}"})

,measure,pandas,dbt,match
0,Bridge: largest difference on any entity and step (€),0.00,0.00,True
1,Group actual EBITDA (€),"4,408,553.06","4,408,553.06",True
2,Group sales price effect (€),"868,077.19","868,077.19",True
3,UK FX translation (€),"-23,901.88","-23,901.88",True
4,Duplicate invoices,1.00,1.00,True
5,Spikes,6.00,6.00,True


Every step of the bridge for every entity matches the dbt mart to the cent, and both checks find the same items.

## Step 6. Publish

* **Web dashboard** ([`web/`](../web), live at [guilhermerigonato.com/finance-fpa-analytics](https://guilhermerigonato.com/finance-fpa-analytics/)):
  [`scripts/export.py`](../scripts/export.py) writes JSON from the marts; clicking an entity, a month, a cost centre
  or a ledger row filters the other panels.
* **Power BI** ([`powerbi/`](../powerbi)): PBIP with a TMDL semantic model, DAX measures for variance and the
  bridge, row-level security by entity, checked with the Best Practice Analyzer rules in [`bpa-rules.json`](../bpa-rules.json).
* **Qlik Sense** ([`qlik/`](../qlik)): load script, set analysis for budget vs actual, section access by entity.

## Which library for interactive charts

| Library | Linked charts | Output | Best for |
|---|---|---|---|
| **Altair** (Vega-Lite), used here | Built in: selections shared across charts | Static HTML, no server | Reports that must open anywhere |
| **Plotly** | Per chart; linking across charts needs Dash | HTML; a Dash app for linking | Rich single charts, maps |
| **Bokeh** | Linked brushing through a shared data source | HTML, or a Bokeh server | Large scatter plots, custom tools |
| **HoloViews + Panel** | `link_selections` in one call | Notebook or Panel app | Turning a notebook into a small app |
| **Dash / Streamlit** | Any, through Python callbacks | Needs a server | Internal tools with live data |